# 農地問題エリア検出システム

**画像ソース**: 国土地理院 全国最新写真（シームレス） - APIキー不要・完全無料

## 実行順序
1. セル1〜3: 初期セットアップ（最初に一度だけ）
2. セル4: 設定（GeoJSONパスを変更）
3. セル5〜9: 画像取得
4. **手動作業**: `data/unlabeled/` の画像を `data/farmland/` と `data/problem/` に振り分け
5. セル10〜13: CNN学習
6. セル14〜16: 推論・自動仕分け
7. **手動作業**: `data/review/` の画像を振り分け
8. セル17: 継続学習（精度が上がるまで 手動仕分け→継続学習 を繰り返す）

In [ ]:
# ===== セル1: ライブラリインストール（初回のみ） =====
!pip install geopandas shapely Pillow torch torchvision tqdm matplotlib requests scikit-learn pandas pyproj ipywidgets mercantile -q
# tqdmのプログレスバーをJupyterで表示するために ipywidgets が必要
# インストール後にカーネルを再起動してください（メニュー: Kernel → Restart）

In [ ]:
# ===== セル2: インポート =====
import hashlib
import io
import math
import shutil
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models

# tqdm: notebook環境で固まる場合は tqdm.auto が自動判別して安全
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {DEVICE}')

In [ ]:
# ===== セル3: ディレクトリ作成 =====
for d in ['data/unlabeled', 'data/farmland', 'data/problem', 'data/review', 'models', 'logs']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('ディレクトリ作成完了')

In [ ]:
# ===== セル4: 設定（ここを自分の環境に合わせて変更） =====
GEOJSON_PATH = 'data/farmland.geojson'  # ← 農地GeoJSONのパスを指定
OUTPUT_DIR   = 'data/unlabeled'
MAX_POLYGONS = 200    # None で全件。まず小さい数でテスト推奨
ZOOM         = 18     # 18=約0.6m/pixel。大きいポリゴンは自動で下げる
OUT_SIZE     = 512    # 保存する画像サイズ（px）
MARGIN_TILES = 1      # ポリゴン周囲の余白タイル数

# 国土地理院タイルURL（APIキー不要・無料）
GSI_TILE_URL = 'https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg'

In [ ]:
# ===== セル5: 画像取得ヘルパー関数 =====
import math
import re
import mercantile

TILE_SIZE = 256  # GSIタイルは256x256px固定

# ---------- ファイル名生成（所在・地番） ----------
# GeoJSONのカラム名に合わせて以下を編集してください
SHOZAI_COL = 'Shozai'   # 所在のカラム名
CHIBAN_COL = 'Chiban'   # 地番のカラム名

def _sanitize_filename(s):
    """ファイル名に使えない文字を除去・置換する。"""
    s = str(s).strip()
    s = s.replace('/', '-').replace('\\', '-').replace(':', '-')
    s = re.sub(r'[<>"|?*\x00-\x1f]', '', s)
    s = s.strip('. ')
    return s[:80]

_filename_seen: set = set()  # セル8で reset() する

def polygon_filename(row, idx, geom):
    """
    所在＋地番からファイル名を組み立てる。
    カラムがない・空の場合は MD5 uid にフォールバック。
    重複時は末尾に _2, _3 ... を付与。
    """
    shozai = row.get(SHOZAI_COL) if SHOZAI_COL in row.index else None
    chiban = row.get(CHIBAN_COL) if CHIBAN_COL in row.index else None

    has_shozai = shozai is not None and pd.notna(shozai) and str(shozai).strip()
    has_chiban = chiban is not None and pd.notna(chiban) and str(chiban).strip()

    if has_shozai and has_chiban:
        base = _sanitize_filename(f'{shozai}_{chiban}')
    elif has_shozai:
        base = _sanitize_filename(f'{shozai}_{idx}')
    else:
        base = hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]

    name = base
    counter = 2
    while name in _filename_seen:
        name = f'{base}_{counter}'
        counter += 1
    _filename_seen.add(name)
    return name

def polygon_uid(geom, idx):
    return hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]

# ---------- タイル座標・ピクセル変換 ----------

def lonlat_to_global_pixel(lon, lat, zoom):
    lat = max(min(lat, 85.05112878), -85.05112878)
    siny = math.sin(math.radians(lat))
    scale = TILE_SIZE * (2 ** zoom)
    gx = (lon + 180.0) / 360.0 * scale
    gy = (0.5 - math.log((1 + siny) / (1 - siny)) / (4 * math.pi)) * scale
    return gx, gy

def lonlat_to_mosaic_pixel(lon, lat, zoom, left_tile_x, top_tile_y):
    gx, gy = lonlat_to_global_pixel(lon, lat, zoom)
    return gx - left_tile_x * TILE_SIZE, gy - top_tile_y * TILE_SIZE

def latlon_to_tile_xy(lat, lon, zoom):
    t = mercantile.tile(lon, lat, zoom)
    return t.x, t.y

# ---------- タイル取得 ----------

def fetch_tile(tx, ty, zoom, session, retries=3):
    url = GSI_TILE_URL.format(z=zoom, x=tx, y=ty)
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code != 200:
                return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (255, 255, 255))
            return Image.open(io.BytesIO(resp.content)).convert('RGB')
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
    return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (255, 255, 255))

# ---------- モザイク合成 ----------

def fetch_region_image(bounds, zoom, margin=1, session=None):
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = requests.Session()
    tiles = list(mercantile.tiles(minx, miny, maxx, maxy, [zoom]))
    if not tiles:
        raise ValueError("No tiles found for bounds")
    xs = [t.x for t in tiles]; ys = [t.y for t in tiles]
    tx_min, tx_max = min(xs) - margin, max(xs) + margin
    ty_min, ty_max = min(ys) - margin, max(ys) + margin
    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE), (255, 255, 255))
    for ty in range(ty_min, ty_max + 1):
        for tx in range(tx_min, tx_max + 1):
            tile = fetch_tile(tx, ty, zoom, session)
            canvas.paste(tile, ((tx - tx_min) * TILE_SIZE, (ty - ty_min) * TILE_SIZE))
    left,  top    = lonlat_to_mosaic_pixel(minx, maxy, zoom, tx_min, ty_min)
    right, bottom = lonlat_to_mosaic_pixel(maxx, miny, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, int(left) - pad),  max(0, int(top) - pad),
                min(canvas.width, int(right) + pad), min(canvas.height, int(bottom) + pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

# ---------- オーバーレイ描画（塗りなし・赤輪郭線 width=1） ----------

def draw_polygon_overlay(img, geom, zoom, origin_tx, origin_ty, crop_box):
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    crop_left, crop_top = crop_box[0], crop_box[1]
    def to_px(lon, lat):
        mx, my = lonlat_to_mosaic_pixel(lon, lat, zoom, origin_tx, origin_ty)
        return mx - crop_left, my - crop_top
    rings = []
    if geom.geom_type == 'Polygon':
        rings = [geom.exterior] + list(geom.interiors)
    elif geom.geom_type == 'MultiPolygon':
        for poly in geom.geoms:
            rings.append(poly.exterior)
            rings.extend(poly.interiors)
    for ring in rings:
        pixels = [to_px(lon, lat) for lon, lat in ring.coords]
        if len(pixels) >= 3:
            draw.polygon(pixels, fill=None)  # 塗りなし
            draw.line(pixels + [pixels[0]], fill=(255, 0, 0, 255), width=1)
    return Image.alpha_composite(img.convert('RGBA'), overlay).convert('RGB')

# ---------- ユーティリティ ----------

def calc_auto_zoom(bounds, max_zoom=18):
    minx, miny, maxx, maxy = bounds
    for z in range(max_zoom, 1, -1):
        tiles = list(mercantile.tiles(minx, miny, maxx, maxy, [z]))
        if len(tiles) <= 16:
            return z
    return 12

print('関数定義完了')
print(f'  ファイル名: 所在カラム="{SHOZAI_COL}", 地番カラム="{CHIBAN_COL}"')
print(f'  ※ カラム名が違う場合は SHOZAI_COL / CHIBAN_COL を書き換えてください')
print(f'  ※ セル7実行後に確認: print(list(gdf.columns))')


In [ ]:
# ===== セル6: 接続テスト（タイル1枚だけ取得・数秒で完了） =====
# 広い範囲をzoom=18で取得すると数百枚になるため、1枚だけで確認する
session = requests.Session()
session.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})

test_lat, test_lon = 36.10, 140.08  # つくば市付近
tx, ty = latlon_to_tile_xy(test_lat, test_lon, zoom=18)
test_tile = fetch_tile(tx, ty, zoom=18, session=session)

plt.figure(figsize=(5, 5))
plt.imshow(test_tile)
plt.title(f'接続テスト OK（zoom=18, 1タイル=256x256px）\n国土地理院 全国最新写真')
plt.axis('off')
plt.show()
print(f'接続OK - タイル座標: z=18, x={tx}, y={ty}')
print('※ 実際の農地画像取得（セル8）ではポリゴン範囲の複数タイルを自動結合します')

In [ ]:
# ===== セル7: GeoJSON読み込み・分布確認 =====
gdf = gpd.read_file(GEOJSON_PATH)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)
if MAX_POLYGONS:
    gdf = gdf.head(MAX_POLYGONS)

print(f'処理対象ポリゴン数: {len(gdf)}')
gdf.plot(figsize=(10, 8), color='green', alpha=0.3, edgecolor='black', linewidth=0.3)
plt.title('農地ポリゴン分布')
plt.show()

In [ ]:
# ===== セル8: 画像取得メインループ（並列化 + 小ポリゴン対応版） =====
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from tqdm import tqdm as tqdm_cli

WORKERS  = 4      # 同時並列数。GSIサーバー負荷軽減のため4以上に上げないこと
MIN_SPAN = 0.0003 # これ未満のポリゴンは中心から拡張して取得（約30m）
MAX_SPAN = 0.5    # これ以上はスキップ（約50km）

# ---------- ヘルパー ----------

def make_session():
    s = requests.Session()
    s.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})
    adapter = HTTPAdapter(pool_connections=WORKERS * 2, pool_maxsize=WORKERS * 4)
    s.mount('https://', adapter)
    return s

def ensure_min_bounds(bounds, min_span=MIN_SPAN):
    """ポリゴンが小さすぎる場合、中心から min_span に広げた bounds を返す。"""
    minx, miny, maxx, maxy = bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    if max(maxx - minx, maxy - miny) < min_span:
        half = min_span / 2
        return (cx - half, cy - half, cx + half, cy + half)
    return bounds

def fetch_region_image_parallel(bounds, zoom, margin=1, session=None):
    """mercantile でタイルリストを作り、並列取得してキャンバスに結合・クロップ。"""
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = make_session()

    tiles = list(mercantile.tiles(minx, miny, maxx, maxy, [zoom]))
    if not tiles:
        raise ValueError("No tiles found for bounds")

    xs = [t.x for t in tiles]; ys = [t.y for t in tiles]
    tx_min, tx_max = min(xs) - margin, max(xs) + margin
    ty_min, ty_max = min(ys) - margin, max(ys) + margin

    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE), (255, 255, 255))

    all_coords = [(tx, ty) for ty in range(ty_min, ty_max + 1)
                            for tx in range(tx_min, tx_max + 1)]

    with ThreadPoolExecutor(max_workers=min(len(all_coords), WORKERS * 2)) as ex:
        future_to_pos = {ex.submit(fetch_tile, tx, ty, zoom, session): (tx, ty)
                         for tx, ty in all_coords}
        for fut in as_completed(future_to_pos):
            tx, ty = future_to_pos[fut]
            tile = fut.result()
            canvas.paste(tile, ((tx - tx_min) * TILE_SIZE, (ty - ty_min) * TILE_SIZE))

    left,  top    = lonlat_to_mosaic_pixel(minx, maxy, zoom, tx_min, ty_min)
    right, bottom = lonlat_to_mosaic_pixel(maxx, miny, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, int(left) - pad),  max(0, int(top) - pad),
                min(canvas.width, int(right) + pad), min(canvas.height, int(bottom) + pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

# ---------- 1ポリゴン処理 ----------

_write_lock = threading.Lock()

def process_polygon(idx, row, output_dir, session):
    geom = row.geometry

    # ファイル名を所在・地番から生成（重複時は _2, _3 ... を付与）
    with _write_lock:
        fname = polygon_filename(row, idx, geom)
    out_path = output_dir / f'{fname}.jpg'

    if out_path.exists():
        return 'exists', None

    bounds = geom.bounds
    span   = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    if span > MAX_SPAN:
        return 'toobig', None

    fetch_bounds = ensure_min_bounds(bounds)
    zoom = calc_auto_zoom(fetch_bounds, max_zoom=ZOOM)

    try:
        img, tx0, ty0, cbox = fetch_region_image_parallel(
            fetch_bounds, zoom=zoom, margin=MARGIN_TILES, session=session)
        img = draw_polygon_overlay(img, geom, zoom, tx0, ty0, cbox)
        img = img.resize((OUT_SIZE, OUT_SIZE), Image.LANCZOS)
        img.save(out_path, 'JPEG', quality=92)
        return 'success', {'filename': fname, 'path': str(out_path), 'zoom': zoom,
                           'span': round(span, 6), 'expanded': span < MIN_SPAN}
    except Exception as e:
        return 'error', str(e)

# ---------- メインループ ----------

output_dir   = Path(OUTPUT_DIR)
session_pool = make_session()

# 重複カウンターをリセット（セル再実行時の二重カウント防止）
_filename_seen.clear()

success = skip_exists = skip_toobig = error = 0
meta_records = []

pbar = tqdm_cli(total=len(gdf), desc='画像取得', unit='件', dynamic_ncols=True, leave=True)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(process_polygon, idx, row, output_dir, session_pool): idx
        for idx, row in gdf.iterrows()
    }
    for fut in as_completed(futures):
        idx = futures[fut]
        status, data = fut.result()
        if status == 'success':
            success += 1
            with _write_lock:
                meta_records.append(data)
        elif status == 'exists':
            skip_exists += 1
        elif status == 'toobig':
            skip_toobig += 1
        else:
            error += 1
            tqdm_cli.write(f'  [error] idx={idx}: {data}')
        pbar.update(1)
        pbar.set_postfix(成功=success, 既存=skip_exists, 大きすぎ=skip_toobig, エラー=error)

pbar.close()

# metadata.csv に追記（既存データとマージ）
meta_path = output_dir / 'metadata.csv'
if meta_path.exists() and meta_records:
    existing = pd.read_csv(meta_path)
    merged = pd.concat([existing, pd.DataFrame(meta_records)]).drop_duplicates('filename')
    merged.to_csv(meta_path, index=False)
elif meta_records:
    pd.DataFrame(meta_records).to_csv(meta_path, index=False)

print(f'\n--- 完了 ---')
print(f'  成功          : {success} 件')
print(f'  既存スキップ   : {skip_exists} 件')
print(f'  大きすぎスキップ: {skip_toobig} 件')
print(f'  エラー        : {error} 件')
print(f'  合計保存済み   : {len(list(output_dir.glob("*.jpg")))} 枚')
if meta_records:
    print(f'\n【ファイル名サンプル（先頭5件）】')
    for r in meta_records[:5]:
        print(f'  {r["filename"]}.jpg')


In [ ]:
# ===== セル9: 取得画像のサムネイル確認（先頭12枚） =====
images  = sorted(output_dir.glob('*.jpg'))[:12]
meta_df = pd.read_csv(output_dir / 'metadata.csv') if (output_dir / 'metadata.csv').exists() else pd.DataFrame()
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flat, images):
    ax.imshow(Image.open(p))
    row = meta_df[meta_df.uid == p.stem] if len(meta_df) else pd.DataFrame()
    z = int(row.zoom.values[0]) if len(row) else '?'
    ax.set_title(f'{p.stem[:8]}\nzoom={z}', fontsize=7)
    ax.axis('off')
for ax in axes.flat[len(images):]:
    ax.axis('off')
plt.suptitle('取得した農地画像（先頭12枚） - 国土地理院 全国最新写真')
plt.tight_layout()
plt.show()

In [ ]:
# ===== セル10: ラベリング状況確認 =====
# 画像取得後、data/unlabeled/ の画像を目視で確認し
# data/farmland/ → 正常な農地
# data/problem/  → 建物・道路が1/3以上の問題エリア
# に手動で振り分けてからこのセルを実行してください（各クラス50枚以上推奨）

for cls in ['farmland', 'problem', 'unlabeled', 'review']:
    n = len(list(Path(f'data/{cls}').glob('*.jpg'))) if Path(f'data/{cls}').exists() else 0
    print(f'  {cls:12s}: {n} 枚')

In [ ]:
# ===== セル11: モデル・学習関数の定義（EfficientNet-B2） =====
DATA_DIR   = '/Users/nk19187/Downloads/datafarm'  # ← 自分の環境に合わせて変更
MODEL_OUT  = 'models/model_niigatav0.pth'
EPOCHS     = 50
BATCH_SIZE = 64
LR         = 1e-4
VAL_RATIO  = 0.15
PATIENCE   = 10  # early stopping

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {DEVICE}')

def build_transforms(train=True):
    if train:
        return T.Compose([
            T.Resize((260, 260)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        T.Resize((260, 260)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def build_model(num_classes=2):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

print('EfficientNet-B2 モデル関数定義完了（入力260×260）')


In [ ]:
# ===== セル12: 初回学習 =====
# ▼ 事前に farmland_list / problem_list を定義しておくこと
#   ファイルは unlabeled/ にあってもOK（自動で探す）

import random
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

PATIENCE = 10  # early stopping

TRAIN_CLASSES = sorted(['farmland', 'problem'])
class_to_idx  = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

# 探索するフォルダ（優先順）
_search_dirs = [
    Path(DATA_DIR) / 'farmland',
    Path(DATA_DIR) / 'problem',
    Path(DATA_DIR) / 'unlabeled',
    Path(DATA_DIR) / 'review',
]

def _resolve_path(p):
    p = Path(p)
    # 1. そのままで存在する
    if p.exists():
        return str(p.resolve())
    # 2. 各フォルダ以下でファイル名一致を探す
    for d in _search_dirs:
        candidate = d / p.name
        if candidate.exists():
            return str(candidate.resolve())
    # 3. 見つからなければそのまま返す（存在確認で除外される）
    return str(p)

samples = (
    [(_resolve_path(p), class_to_idx['farmland']) for p in farmland_list] +
    [(_resolve_path(p), class_to_idx['problem'])  for p in problem_list]
)

# 存在確認
missing = [p for p, _ in samples if not Path(p).exists()]
if missing:
    print(f'⚠️ 見つからなかったファイル: {len(missing)}件')
    for m in missing[:5]:
        print(f'  {m}')
    samples = [(p, lbl) for p, lbl in samples if Path(p).exists()]

random.seed(42)
random.shuffle(samples)

counts = [0, 0]
for _, lbl in samples:
    counts[lbl] += 1

print(f'farmland_list: {len(farmland_list)}件 → 有効: {counts[class_to_idx["farmland"]]}枚')
print(f'problem_list : {len(problem_list)}件 → 有効: {counts[class_to_idx["problem"]]}枚')

if 0 in counts:
    missing_cls = [TRAIN_CLASSES[i] for i, c in enumerate(counts) if c == 0]
    raise ValueError(f'⚠️ {missing_cls} に画像がありません')

# --- SimpleImageDataset ---
class SimpleImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# --- train/val 分割 ---
n_val   = max(1, int(len(samples) * VAL_RATIO))
n_train = len(samples) - n_val
train_ds = SimpleImageDataset(samples[:n_train], transform=build_transforms(train=True))
val_ds   = SimpleImageDataset(samples[n_train:], transform=build_transforms(train=False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'train={n_train}枚, val={n_val}枚')

# --- クラス重み・モデル・最適化 ---
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
model     = build_model(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history, best_val_acc, no_improve = [], 0.0, 0
Path(MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, EPOCHS + 1), desc='学習'):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    tqdm.write(f'Epoch {epoch:03d} | train={train_acc:.4f} loss={train_loss:.4f} | val={val_acc:.4f} loss={val_loss:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'class_to_idx': class_to_idx, 'val_acc': val_acc}, MODEL_OUT)
        tqdm.write(f'  → モデル保存 (val_acc={val_acc:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            tqdm.write(f'Early stopping (patience={PATIENCE})')
            break

print(f'\n学習完了。最良 val_acc={best_val_acc:.4f}')

In [ ]:
# ===== セル13: 学習曲線グラフ =====
df_hist = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_hist.epoch, df_hist.train_loss, label='train')
ax1.plot(df_hist.epoch, df_hist.val_loss,   label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(df_hist.epoch, df_hist.train_acc, label='train')
ax2.plot(df_hist.epoch, df_hist.val_acc,   label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('logs/training_curve.png', dpi=150)
plt.show()

In [ ]:
# ===== セル14: 推論設定 =====
PREDICT_MODEL   = 'models/model_niigatav0.pth'   # 使用するモデル
PREDICT_INPUT   = Path(DATA_DIR) / 'unlabeled'   # 推論対象フォルダ
THRESHOLD_PROB  = 0.85   # problem判定の最低確信度（これ未満はreviewへ）
THRESHOLD_FARM  = 0.5    # farmland判定の最低確信度
BATCH_SIZE_INFER = 64    # B2 + MPS: 64推奨
DRY_RUN         = True   # True=移動しない（確認用）。False で実際に移動

print(f'推論モデル  : {PREDICT_MODEL}')
print(f'推論対象    : {PREDICT_INPUT}')
print(f'THRESHOLD_PROB(problem) : {THRESHOLD_PROB}')
print(f'THRESHOLD_FARM(farmland): {THRESHOLD_FARM}')
print(f'DRY_RUN     : {DRY_RUN}')


In [ ]:
# ===== セル15: 推論実行（バッチ推論・高速版） =====
from torch.utils.data import Dataset, DataLoader

def _build_model_for_infer(num_classes=2):
    """チェックポイントのアーキテクチャを自動判別してモデルを返す。"""
    ckpt = torch.load(PREDICT_MODEL, map_location='cpu')
    # features.1.1 の有無でB0/B2を判別（B2のみfeatures.1に2ブロック存在）
    if 'features.1.1.block.0.0.weight' in ckpt['model']:
        m = models.efficientnet_b2(weights=None)
        print('アーキテクチャ自動判別: B2')
    else:
        m = models.efficientnet_b0(weights=None)
        print('アーキテクチャ自動判別: B0')
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m, ckpt

pred_model, state = _build_model_for_infer()
pred_model = pred_model.to(DEVICE)
pred_model.load_state_dict(state['model'])
pred_model.eval()
idx_to_class = {v: k for k, v in state['class_to_idx'].items()}
print(f'モデル読み込み完了 (val_acc={state["val_acc"]:.4f})')

infer_transform = build_transforms(train=False)

class InferDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        try:
            img = Image.open(p).convert('RGB')
            return self.transform(img), str(p)
        except Exception:
            return torch.zeros(3, 260, 260), str(p)

input_dir = Path(PREDICT_INPUT)
out_dirs  = {
    'farmland': Path(DATA_DIR) / 'farmland',
    'problem':  Path(DATA_DIR) / 'problem',
    'review':   Path(DATA_DIR) / 'review',
}
if not DRY_RUN:
    for d in out_dirs.values():
        d.mkdir(parents=True, exist_ok=True)

all_images = sorted(input_dir.glob('*.jpg'))
print(f'推論対象: {len(all_images)}枚')

infer_ds     = InferDataset(all_images, infer_transform)
infer_loader = DataLoader(infer_ds, batch_size=BATCH_SIZE_INFER, shuffle=False,
                          num_workers=0, pin_memory=False)

records = []
with torch.no_grad():
    for batch_imgs, batch_paths in tqdm(infer_loader, desc='推論'):
        batch_imgs = batch_imgs.to(DEVICE)
        probs_all  = F.softmax(pred_model(batch_imgs), dim=1)
        for probs, img_path in zip(probs_all, batch_paths):
            farm_prob = probs[class_to_idx['farmland']].item()
            prob_prob = probs[class_to_idx['problem']].item()
            if prob_prob >= THRESHOLD_PROB:
                pred_class, confidence, dest = 'problem', prob_prob, 'problem'
            elif farm_prob >= THRESHOLD_FARM:
                pred_class, confidence, dest = 'farmland', farm_prob, 'farmland'
            else:
                pred_class = 'farmland' if farm_prob >= prob_prob else 'problem'
                confidence = max(farm_prob, prob_prob)
                dest = 'review'
            records.append({'file': Path(img_path).name, 'pred': pred_class,
                            'confidence': round(confidence, 4),
                            'farm_prob': round(farm_prob, 4),
                            'prob_prob': round(prob_prob, 4),
                            'dest': dest})
            if not DRY_RUN:
                shutil.move(img_path, out_dirs[dest] / Path(img_path).name)

df_pred = pd.DataFrame(records)
df_pred.to_csv(Path(DATA_DIR) / 'predict_report.csv', index=False)
print('\n--- 推論結果 ---')
print(df_pred.dest.value_counts().to_string())
print(f'\n平均確信度: {df_pred.confidence.mean():.4f}')
if DRY_RUN:
    print('\n※ DRY_RUN=True のため移動していません。False にして再実行してください。')

In [ ]:
# ===== セル16: 確信度分布グラフ + review画像サムネイル =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pred.confidence, bins=30, edgecolor='black')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', label=f'threshold={THRESHOLD}')
axes[0].set_xlabel('確信度'); axes[0].set_ylabel('件数')
axes[0].set_title('確信度分布'); axes[0].legend()
df_pred.dest.value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('振り分け結果'); axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

review_images = sorted(Path('data/review').glob('*.jpg'))[:16]
if review_images:
    cols = 4
    rows = (len(review_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    for ax, p in zip(axes.flat, review_images):
        row = df_pred[df_pred.file == p.name]
        conf = row.confidence.values[0] if len(row) else 0
        pred = row.pred.values[0] if len(row) else '?'
        ax.imshow(Image.open(p))
        ax.set_title(f'pred={pred}\nconf={conf:.2f}', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(review_images):]:
        ax.axis('off')
    plt.suptitle('要確認画像（review/）→ farmland/ or problem/ に手動で移動してください')
    plt.tight_layout()
    plt.show()
else:
    print('review/ に画像がありません')

In [ ]:
# ===== セル16b: 誤判定画像を corrected/ に移動 =====
import glob
import shutil
from pathlib import Path

DATA_DIR = '/Users/masashi-ijichi/Downloads/datafarm2'  # ← 自分の環境に合わせて変更

wrong_as_farmland = [
    '長野県信濃町柏原瑞穂4974-1_4974-1.jpg','長野県山ノ内町夜間瀬土平3980-1_3980-1.jpg','長野県木島平村大字上木島字池平3277-42_3277-42.jpg',
    '長野県木島平村大字往郷字荒川1796-1_1796-1.jpg','長野県山ノ内町平穏上川原174-2_174-2.jpg','長野県須坂市大字日滝地蔵原3969-1_3969-1.jpg',
    '長野県木島平村大字往郷字千石3275-2_3275-2.jpg','長野県須坂市大字日滝古池2461-1_2461-1.jpg','長野県飯綱町黒川浦山1148-1_1148-1.jpg',
    '長野県木島平村大字上木島字池平3277-42_3277-42.jpg','長野県山ノ内町平穏上川原174-2_174-2.jpg','長野県木島平村大字往郷字千石3275-2_3275-2.jpg',
    '長野県飯綱町東柏原原1110-1_1110-1.jpg','長野県小布施町飯田木原127-1(０)_127-1(０).jpg','長野県須坂市大字小河原別府組沖2374-1_2374-1.jpg',
    '長野県小布施町飯田庚申塔728(０)_728(０).jpg','長野県飯綱町東柏原原1105-1_1105-1.jpg','新潟県糸魚川市中川原新田前田2141-1_2141-1.jpg',
    '長野県信濃町富濃西脇3001-1_3001-1.jpg','長野県飯綱町赤塩北村6378-1_6378-1.jpg','長野県小布施町飯田中村屋敷845-2(０)_845-2(０).jpg',
    '長野県飯綱町東柏原原1112_1112.jpg',
]

wrong_as_problem = [
    '新潟県湯沢町土樽向原6626_6626.jpg','長野県飯綱町倉井今田628-62_628-62.jpg','長野県飯綱町倉井大久保4507-1_4507-1.jpg',
    '長野県小布施町飯田三左エ門屋敷222-3(０)_222-3(０).jpg','新潟県南魚沼市上一日市神代免496-1_496-1.jpg','長野県山ノ内町寒沢入原1775_1775.jpg',
    '長野県小布施町中松東畑375-1(０)_375-1(０).jpg','長野県山ノ内町平穏欠端4287-1_4287-1.jpg','長野県小布施町北岡下り134-1(０)_134-1(０).jpg',
    '新潟県糸魚川市中川原新田上中川原2579-1_2579-1.jpg','長野県須坂市大字日滝丹波塚4318-1_4318-1.jpg','長野県須坂市大字小河原新田組沖2556_2556.jpg',
    '長野県須坂市大字沼目北沖586-2_586-2.jpg','長野県小布施町飯田山王島大道下263(０)_263(０).jpg','長野県飯綱町赤塩曲坂2162-1_2162-1.jpg',
    '長野県小布施町山王島飯田境21-1(０)_21-1(０).jpg','長野県木島平村大字往郷字上原3032_3032.jpg','長野県須坂市大字相之島新田1391-1_1391-1.jpg',
    '長野県飯綱町東柏原前田118_118.jpg','長野県木島平村大字往郷字千石3352-1_3352-1.jpg','長野県飯綱町東柏原石畑2517_2517.jpg',
    '長野県須坂市大字小河原六川道西沖3215_3215.jpg','長野県小布施町小布施北畑1776-3(０)_1776-3(０).jpg','長野県信濃町柏原柏原4355-1_4355-1.jpg',
    '長野県信濃町柏原瑞穂4975-1_4975-1.jpg','長野県信濃町柏原谷地2252-1_2252-1.jpg','長野県飯綱町赤塩毛見1212-5_1212-5.jpg',
    '長野県須坂市大字小河原新田組沖2960_2960.jpg','長野県小布施町雁田中道1044-1(０)_1044-1(０).jpg','長野県信濃町大井霊仙寺2726-2_2726-2.jpg',
    '長野県山ノ内町平穏■本5060-2_5060-2.jpg','長野県小布施町大島切次752-36(０)_752-36(０).jpg','長野県小布施町北岡西■坊529-1(０)_529-1(０).jpg',
    '新潟県糸魚川市中川原新田村田1810-1_1810-1.jpg','長野県飯綱町倉井大峰1191-2_1191-2.jpg','長野県小布施町雁田清水端670-1(０)_670-1(０).jpg',
    '長野県小布施町小布施北畑1829-1(０)_1829-1(０).jpg','長野県小布施町都住道添832-ロ(０)_832-ロ(０).jpg','長野県小布施町小布施林2221(０)_2221(０).jpg',
    '長野県小布施町飯田庚申塔697-1(０)_697-1(０).jpg','長野県小布施町都住六川沖西1189-1(０)_1189-1(０).jpg','長野県須坂市大字日滝古池2493-1_2493-1.jpg',
    '長野県小布施町押羽西郷704(０)_704(０).jpg','長野県小布施町飯田宮南1188-45(０)_1188-45(０).jpg','長野県山ノ内町夜間瀬天久保2025-2_2025-2.jpg',
    '長野県信濃町古海菅川4207-1_4207-1.jpg','長野県須坂市大字小河原六川道西沖3026-1_3026-1.jpg','長野県飯綱町平出西浦1027-2_1027-2.jpg',
    '長野県小布施町飯田古屋敷512(０)_512(０).jpg','長野県須坂市大字小島柳原155-1_155-1.jpg','長野県小布施町小布施唐澤2452-28(０)_2452-28(０).jpg',
    '長野県小布施町雁田西原889(０)_889(０).jpg','長野県小布施町雁田中道1072-イ(０)_1072-イ(０).jpg','長野県山ノ内町平穏欠端4283_4283.jpg',
    '長野県小布施町押羽砂川90(０)_90(０).jpg','長野県小布施町都住六川沖西1254(０)_1254(０).jpg','長野県飯綱町東柏原堤尻469-1_469-1.jpg',
    '長野県飯綱町赤塩柳原6547-1_6547-1.jpg','長野県飯綱町赤塩西原771-5_771-5.jpg','長野県須坂市大字日滝地蔵原4113-4_4113-4.jpg',
    '長野県小布施町押羽新引向977-6(０)_977-6(０).jpg','長野県小布施町都住三毛315-5(０)_315-5(０).jpg','長野県木島平村大字往郷字小路2842-1_2842-1.jpg',
    '長野県須坂市大字小島北ノ原1470-3_1470-3.jpg','長野県小布施町北岡下り74-1(０)_74-1(０).jpg','長野県飯綱町東柏原前田141_141.jpg',
    '長野県須坂市大字相之島新田1402_1402.jpg','長野県飯綱町赤塩曽峯1095_1095.jpg','長野県木島平村大字穂高字北鴨境1393-2_1393-2.jpg',
    '長野県飯綱町東柏原向田3020-1_3020-1.jpg','長野県飯綱町袖之山成合680_680.jpg','長野県山ノ内町平穏夜間瀬境4728-1_4728-1.jpg',
    '長野県小布施町小布施林1924(０)_1924(０).jpg','長野県須坂市大字日滝虫送3579_3579.jpg','長野県小布施町飯田中村屋敷851(０)_851(０).jpg',
    '長野県飯綱町普光寺東原860-1_860-1.jpg','新潟県湯沢町神立上戸沢2793-2_2793-2.jpg','長野県小布施町小布施林2182-1(０)_2182-1(０).jpg',
    '新潟県十日町市清田山己1950-1_己1950-1.jpg','長野県山ノ内町佐野谷地679-5_679-5.jpg','長野県飯綱町東柏原塔場362_362.jpg',
    '長野県小布施町雁田清水端659(０)_659(０).jpg','長野県須坂市大字八重森西久保174_174.jpg','長野県山ノ内町佐野宮下260-2_260-2.jpg',
    '長野県小布施町福原久保140-2(０)_140-2(０).jpg','長野県須坂市大字相之島大日堂787-1_787-1.jpg','長野県木島平村大字往郷字屋敷添4239_4239.jpg',
    '長野県木島平村大字穂高字大原2482-1_2482-1.jpg','長野県飯綱町倉井大畑3029-1_3029-1.jpg','長野県飯綱町東柏原原1034-イ_1034-イ.jpg',
    '長野県飯綱町牟礼城山2251_2251.jpg','長野県小布施町押羽東柳原1557-10(０)_1557-10(０).jpg','長野県木島平村大字往郷字荒川1782-1_1782-1.jpg',
    '長野県須坂市大字小河原六川道西沖3040-1_3040-1.jpg','長野県須坂市大字相之島上河原722-1_722-1.jpg','新潟県南魚沼市仙石528-1_528-1.jpg',
    '長野県飯綱町赤塩里久保3276_3276.jpg','長野県飯綱町倉井大峰1238-1_1238-1.jpg','長野県須坂市大字日滝境塚3215-1_3215-1.jpg',
    '長野県木島平村大字穂高字三枚原849-17_849-17.jpg','新潟県糸魚川市中川原新田3039-1_3039-1.jpg','長野県飯綱町川上中夏川1100-1_1100-1.jpg',
    '長野県飯綱町倉井今田628-54_628-54.jpg','長野県須坂市大字日滝丹波塚4572_4572.jpg','長野県須坂市大字小河原柳沢東沖232-3_232-3.jpg',
    '長野県飯綱町赤塩日影林5838_5838.jpg','長野県信濃町柏原西原3543-1_3543-1.jpg','新潟県糸魚川市中川原新田上中川原2560_2560.jpg',
    '長野県須坂市大字小島前田187-1_187-1.jpg','長野県山ノ内町夜間瀬東町2792-1_2792-1.jpg','長野県須坂市大字相之島裏河原1087_1087.jpg',
    '長野県小布施町都住狐■567(０)_567(０).jpg','長野県飯綱町芋川向山1177_1177.jpg','長野県信濃町野尻御小屋1390-2_1390-2.jpg',
    '長野県飯綱町東柏原原1077-2_1077-2.jpg','長野県飯綱町赤塩中毛野2670_2670.jpg','長野県飯綱町赤塩川手3765-1_3765-1.jpg',
    '長野県飯綱町古町下向山51-1_51-1.jpg','長野県須坂市大字日滝虫送3453-1_3453-1.jpg','長野県小布施町北岡東屋敷添481-4(０)_481-4(０).jpg',
    '長野県飯綱町芋川向山1153_1153.jpg','長野県飯綱町倉井大久保4576-1_4576-1.jpg','長野県信濃町大井霊仙寺2742-69_2742-69.jpg',
    '長野県小布施町中松東側屋敷295(０)_295(０).jpg','長野県須坂市大字小河原北組沖1482-1_1482-1.jpg','長野県須坂市大字小島欠下668-5_668-5.jpg',
    '長野県信濃町大井霊仙寺2742-64_2742-64.jpg','長野県木島平村大字往郷字上原3031-1_3031-1.jpg','長野県小布施町雁田馬場先743-1(０)_743-1(０).jpg',
    '長野県小布施町雁田堰下897-1(０)_897-1(０).jpg','長野県小布施町中松古宮429-3(０)_429-3(０).jpg','長野県小布施町福原久保134-5(０)_134-5(０).jpg',
    '長野県山ノ内町平穏中道間678-2_678-2.jpg','長野県飯綱町東柏原原1086_1086.jpg','長野県須坂市大字小河原六川道東沖3871_3871.jpg',
    '長野県小布施町押羽恵田985-3(０)_985-3(０).jpg','長野県飯綱町赤塩前坂5559-1_5559-1.jpg','長野県山ノ内町夜間瀬柳原4280-1_4280-1.jpg',
    '長野県小布施町雁田観音崎1202-1(０)_1202-1(０).jpg','長野県飯綱町芋川山王林4994_4994.jpg','長野県飯綱町普光寺立野1806_1806.jpg',
    '新潟県南魚沼市滝谷133-甲_133-甲.jpg','長野県飯綱町芋川土井尻226-1_226-1.jpg','長野県信濃町柏原渋田2730-1_2730-1.jpg',
    '長野県須坂市大字日滝丹波塚4638-2_4638-2.jpg','長野県飯綱町東柏原桑ノ木867_867.jpg','長野県須坂市大字沼目西沖100-4_100-4.jpg',
    '長野県信濃町柏原中原1341_1341.jpg','長野県小布施町小布施林2136(０)_2136(０).jpg','長野県飯綱町東柏原馬放場1508_1508.jpg',
    '長野県須坂市大字小河原新田組沖2484_2484.jpg','長野県小布施町押羽西郷697-イ(０)_697-イ(０).jpg','長野県信濃町大井霊仙寺2742-131_2742-131.jpg',
    '長野県須坂市大字相之島裏河原1048-3_1048-3.jpg','長野県小布施町北岡下り138-1(０)_138-1(０).jpg','長野県信濃町平岡裏屋敷添1684_1684.jpg',
    '長野県小布施町福原屋敷74-1(０)_74-1(０).jpg','長野県山ノ内町寒沢円生里1110-1_1110-1.jpg','長野県小布施町小布施吉島2888-2(０)_2888-2(０).jpg',
    '長野県信濃町熊坂原173-1_173-1.jpg','長野県小布施町小布施林2209(０)_2209(０).jpg','長野県飯綱町赤塩針ノ木2958_2958.jpg',
    '長野県飯綱町東柏原馬場2251-1_2251-1.jpg','長野県小布施町中松雁田境450-2(０)_450-2(０).jpg','長野県須坂市大字相之島開田1308-1_1308-1.jpg',
    '長野県山ノ内町夜間瀬越巻4191-1_4191-1.jpg','長野県須坂市大字小河原南組西沖93-3_93-3.jpg','長野県飯綱町倉井今田605-1_605-1.jpg',
    '長野県須坂市大字小島東田588-2_588-2.jpg','長野県飯綱町東柏原馬場2236_2236.jpg','新潟県糸魚川市大平前田6672_6672.jpg',
    '長野県小布施町大島切次771-4(０)_771-4(０).jpg','長野県小布施町押羽砂川87(０)_87(０).jpg','長野県小布施町小布施林2076-1(０)_2076-1(０).jpg',
    '長野県木島平村大字穂高字原1330_1330.jpg','長野県飯綱町赤塩中毛野2671_2671.jpg','長野県山ノ内町戸狩裏川原1187-1_1187-1.jpg',
    '長野県山ノ内町夜間瀬東町2870_2870.jpg','長野県須坂市大字日滝寺窪2761_2761.jpg','長野県飯綱町東柏原前田128_128.jpg',
    '長野県小布施町飯田中道下633(０)_633(０).jpg','長野県小布施町都住遠徳2427-1(０)_2427-1(０).jpg','長野県飯綱町赤塩川手3771_3771.jpg',
    '新潟県糸魚川市中川原新田村田1741_1741.jpg','長野県小布施町小布施吉島2868-19(０)_2868-19(０).jpg','長野県須坂市大字村山土手外592_592.jpg',
    '長野県須坂市大字相之島裏河原1075-1_1075-1.jpg','長野県飯綱町牟礼本塚1074-3_1074-3.jpg','長野県飯綱町芋川町浦898-2_898-2.jpg',
    '新潟県糸魚川市土倉滝脇川原854-4_854-4.jpg','長野県須坂市大字沼目北沖583-3_583-3.jpg','新潟県糸魚川市大平ササクラ7726_7726.jpg',
    '長野県飯綱町東柏原前田145-ロ_145-ロ.jpg','長野県須坂市大字相之島裏河原1086-2_1086-2.jpg','長野県須坂市大字小河原前田沖74-7_74-7.jpg',
]

def find_image(fname):
    """DATA_DIR以下を再帰検索してファイルを見つける"""
    results = glob.glob(f'{DATA_DIR}/**/{fname}', recursive=True)
    return Path(results[0]) if results else None

# corrected/ ディレクトリ作成
for d in ['corrected/farmland', 'corrected/problem']:
    (Path(DATA_DIR) / d).mkdir(parents=True, exist_ok=True)

moved = 0
not_found = []

# farmlandと判定されたが実はproblem → corrected/problem へ
for fname in wrong_as_farmland:
    src = find_image(fname)
    if src:
        dst = Path(DATA_DIR) / 'corrected' / 'problem' / fname
        shutil.copy2(str(src), str(dst))
        moved += 1
    else:
        not_found.append(('wrong_as_farmland', fname))

# problemと判定されたが実はfarmland → corrected/farmland へ
for fname in wrong_as_problem:
    src = find_image(fname)
    if src:
        dst = Path(DATA_DIR) / 'corrected' / 'farmland' / fname
        shutil.copy2(str(src), str(dst))
        moved += 1
    else:
        not_found.append(('wrong_as_problem', fname))

print(f'\n=== 結果 ===')
print(f'{moved}枚をcorrected/にコピー完了')
print(f'  corrected/problem  (実はproblem):  {len([f for f in wrong_as_farmland if find_image(f)])}枚')
print(f'  corrected/farmland (実はfarmland): {len([f for f in wrong_as_problem if find_image(f)])}枚')
if not_found:
    print(f'\n見つからなかったファイル ({len(not_found)}件):')
    for category, f in not_found:
        print(f'  [{category}] {f}')

In [ ]:
# ===== セル17: 継続学習（corrected/farmland重み付き + pseudo-label + backbone凍結） =====
import random
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

BASE_MODEL            = 'models/model_niigatav0.pth'
NEW_MODEL_OUT         = 'models/model_niigatav1.pth'
RETRAIN_EPOCHS        = 50
RETRAIN_LR            = 2e-5
PATIENCE              = 10
CORRECTED_WEIGHT_FARM = 5   # corrected/farmland の重み倍率
PSEUDO_THRESHOLD      = 0.95
PSEUDO_MAX            = 5000

TRAIN_CLASSES = sorted(['farmland', 'problem'])
class_to_idx  = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

class SimpleImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

def _build_model_for_load(ckpt, num_classes=2):
    """B0/B2をブロック数で判別（features.1.1はB2のみ存在、B0にはない）。"""
    if 'features.1.1.block.0.0.weight' in ckpt['model']:
        m = models.efficientnet_b2(weights=None)
        print('アーキテクチャ自動判別: B2')
    else:
        m = models.efficientnet_b0(weights=None)
        print('アーキテクチャ自動判別: B0')
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

# ファイル探索
_search_dirs = [
    Path(DATA_DIR) / 'farmland',
    Path(DATA_DIR) / 'problem',
    Path(DATA_DIR) / 'unlabeled',
    Path(DATA_DIR) / 'review',
]
def _resolve_path(p):
    p = Path(p)
    if p.exists():
        return str(p.resolve())
    for d in _search_dirs:
        candidate = d / p.name
        if candidate.exists():
            return str(candidate.resolve())
    return str(p)

# ===== Phase 1: farmland_list / problem_list =====
samples = (
    [(_resolve_path(p), class_to_idx['farmland']) for p in farmland_list] +
    [(_resolve_path(p), class_to_idx['problem'])  for p in problem_list]
)
samples = [(p, lbl) for p, lbl in samples if Path(p).exists()]

# ===== Phase 2: corrected/farmland（重み付き） =====
cd_farm = Path(DATA_DIR) / 'corrected' / 'farmland'
corrected_f = 0
if cd_farm.exists():
    imgs = list(cd_farm.glob('*.jpg'))
    corrected_f = len(imgs)
    for p in imgs:
        for _ in range(CORRECTED_WEIGHT_FARM):
            samples.append((str(p), class_to_idx['farmland']))

# ===== Phase 3: pseudo-label（高確信度farmland） =====
report_path = Path(DATA_DIR) / 'predict_report.csv'
pseudo_count = 0
if report_path.exists():
    df_report = pd.read_csv(report_path)
    prob_col = 'farm_prob' if 'farm_prob' in df_report.columns else 'confidence'
    pseudo_df = df_report[
        (df_report.dest == 'farmland') & (df_report[prob_col] >= PSEUDO_THRESHOLD)
    ].head(PSEUDO_MAX)
    for _, row in pseudo_df.iterrows():
        p = _resolve_path(row['file'])
        if Path(p).exists():
            samples.append((p, class_to_idx['farmland']))
            pseudo_count += 1
    print(f'pseudo-label farmland: {pseudo_count}枚追加（{prob_col}>={PSEUDO_THRESHOLD}）')

# ===== 集計 =====
counts = [0, 0]
for _, lbl in samples:
    counts[lbl] += 1

print(f'学習サンプル数: {len(samples)}')
for cls in TRAIN_CLASSES:
    print(f'  {cls}: {counts[class_to_idx[cls]]}枚')
print(f'  (corrected/farmland={corrected_f}枚 x{CORRECTED_WEIGHT_FARM}倍, pseudo={pseudo_count}枚)')

if 0 in counts:
    missing_cls = [TRAIN_CLASSES[i] for i, c in enumerate(counts) if c == 0]
    raise ValueError(f'⚠️ {missing_cls} に画像がありません')

# ===== データローダー =====
random.seed(42)
random.shuffle(samples)
n_val   = max(1, int(len(samples) * VAL_RATIO))
n_train = len(samples) - n_val
train_ds = SimpleImageDataset(samples[:n_train], transform=build_transforms(train=True))
val_ds   = SimpleImageDataset(samples[n_train:], transform=build_transforms(train=False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'train={n_train}枚, val={n_val}枚')

# ===== モデル読み込み（アーキテクチャ自動判別） =====
retrain_state = torch.load(BASE_MODEL, map_location=DEVICE)
retrain_model = _build_model_for_load(retrain_state, num_classes=2).to(DEVICE)
retrain_model.load_state_dict(retrain_state['model'])
print(f'起点モデル val_acc={retrain_state["val_acc"]:.4f}')

# ===== backbone凍結（最終2ブロック + classifier のみ学習） =====
for param in retrain_model.parameters():
    param.requires_grad = False
for block in list(retrain_model.features)[-2:]:
    for param in block.parameters():
        param.requires_grad = True
for param in retrain_model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in retrain_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in retrain_model.parameters())
print(f'学習パラメータ: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# ===== 学習 =====
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, retrain_model.parameters()),
    lr=RETRAIN_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RETRAIN_EPOCHS)

best_r, no_improve = 0.0, 0
Path(NEW_MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, RETRAIN_EPOCHS + 1), desc='継続学習'):
    _, train_acc_r = run_epoch(retrain_model, train_loader, criterion, optimizer)
    _, val_acc_r   = run_epoch(retrain_model, val_loader,   criterion)
    scheduler.step()
    tqdm.write(f'Epoch {epoch:03d} | train={train_acc_r:.4f} | val={val_acc_r:.4f}')
    if val_acc_r >= best_r:
        best_r     = val_acc_r
        no_improve = 0
        torch.save({'epoch': epoch, 'model': retrain_model.state_dict(),
                    'class_to_idx': class_to_idx, 'val_acc': val_acc_r}, NEW_MODEL_OUT)
        tqdm.write(f'  → 保存: {NEW_MODEL_OUT} (val_acc={val_acc_r:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            tqdm.write(f'Early stopping (patience={PATIENCE})')
            break

print(f'\n継続学習完了: {retrain_state["val_acc"]:.4f} → {best_r:.4f}')
print(f'次回: PREDICT_MODEL と BASE_MODEL を {NEW_MODEL_OUT} に変更して再実行')